## Import libraries

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp

## Load Dataset

In [3]:
df = pd.read_csv("data/Fuse.csv", delimiter=';')
print(f"Total entries: {len(df)}")
print(df.columns)

Total entries: 998
Index(['PART_ID', 'PART_DESCRIPTION', 'Acting', 'Additional Feature',
       'Application', 'Blow Characteristic', 'Body Breadth (mm)',
       'Body Height (mm)', 'Body Length or Diameter (mm)', 'Current Rating',
       'Fuse Material', 'Fuse Size', 'JESD-609 Code', 'Joule-integral-Nom (J)',
       'LC Risk', 'Maximum AC Voltage Rating', 'Maximum DC Voltage Rating',
       'Maximum Power Dissipation', 'Mounting', 'Mounting Feature',
       'Number of Terminals', 'Operating Temperature-Max (Cel)',
       'Operating Temperature-Min (Cel)', 'Physical Dimension',
       'Pre-arcing time-Min (ms)', 'Product Diameter', 'Product Length',
       'Rated Breaking Capacity (A)', 'Rated Current (A)', 'Rated Voltage (V)',
       'Rated Voltage(AC) (V)', 'Rated Voltage(DC) (V)'],
      dtype='object')


## Data Inspection

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 998 entries, 0 to 997
Data columns (total 32 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   PART_ID                          998 non-null    object 
 1   PART_DESCRIPTION                 663 non-null    object 
 2   Acting                           770 non-null    object 
 3   Additional Feature               324 non-null    object 
 4   Application                      821 non-null    object 
 5   Blow Characteristic              672 non-null    object 
 6   Body Breadth (mm)                417 non-null    object 
 7   Body Height (mm)                 705 non-null    object 
 8   Body Length or Diameter (mm)     705 non-null    object 
 9   Current Rating                   834 non-null    object 
 10  Fuse Material                    771 non-null    object 
 11  Fuse Size                        880 non-null    object 
 12  JESD-609 Code         

In [7]:
print("\n--- Descriptive Summary of Columns (Top 5 values and Missing Counts) ---")
# Analyze data types, missing values, and sample values
missing_data = df.isnull().sum()
summary_rows = []

for col in df.columns:
    summary_rows.append({
        'Column': col,
        'Dtype': df[col].dtype,
        'Missing Count': missing_data[col],
        'Missing Percent': f"{(missing_data[col] / len(df) * 100):.1f}%",
        'Sample Values': ', '.join(df[col].dropna().head(3).astype(str).tolist())
    })

print(pd.DataFrame(summary_rows).to_markdown(index=False))


--- Descriptive Summary of Columns (Top 5 values and Missing Counts) ---
| Column                          | Dtype   |   Missing Count | Missing Percent   | Sample Values                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:--------------------------------|:--------|----------------:|:------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**Top 2 findings**
> - Finding 1: Inconsistent Data Types and Embedded Units
>   - Most attributes that represent continuous numerical values are stored as object type and include units.
>   - **Difficulty** - These columns cannot be used directly in any distance-based similarity calculation until they are converted to a standard numeric type.
> - Finding 2: High Proportion of Missing Values (Sparsity)
>   - Several columns that are crucial for similarity modeling have a high percentage of missing data.
>     - Pre-arcing time-Min (ms): $\approx 89\%$ missing
>     - Maximum Power Dissipation: $\approx 75\%$ missing
>     - PART_DESCRIPTION: $\approx 33\%$ missing (335 missing entries out of 998)
>   - **Difficulty** - Simply dropping rows with missing values would reduce the dataset size significantly, resulting in a poor or non-existent model.

## Potential Solutions

### Regex Cleaning

Use regular expressions (re) to extract the first continuous number (float or integer) from the string, then convert the column to float.

In [ ]:
# Finding 1: Inconsistent Data Types and Embedded Units
numeric_cols_to_clean = [
    'Body Breadth (mm)', 'Body Height (mm)', 'Body Length or Diameter (mm)',
    'Current Rating', 'Joule-integral-Nom (J)', 'Maximum AC Voltage Rating',
    'Maximum DC Voltage Rating', 'Maximum Power Dissipation',
    'Operating Temperature-Max (Cel)', 'Operating Temperature-Min (Cel)',
    'Pre-arcing time-Min (ms)', 'Product Diameter', 'Product Length',
    'Rated Breaking Capacity (A)', 'Rated Current (A)', 'Rated Voltage (V)',
    'Rated Voltage(AC) (V)', 'Rated Voltage(DC) (V)'
]

def clean_numeric(series):
    """Function to extract numbers from strings using regex."""
    def extract_num(text):
        if pd.isna(text): return np.nan
        # Regex to find the first number (int or float) in the string
        matches = re.findall(r'[-+]?\d*\.\d+|\d+', str(text))
        return float(matches[0]) if matches else np.nan
    return series.apply(extract_num)

df_cleaned = df.copy()
numeric_features = []

for col in numeric_cols_to_clean:
    new_col_name = col + '_NUM'
    df_cleaned[new_col_name] = clean_numeric(df_cleaned[col])
    numeric_features.append(new_col_name)

In [11]:
df_cleaned[numeric_features].dtypes

Body Breadth (mm)_NUM                  float64
Body Height (mm)_NUM                   float64
Body Length or Diameter (mm)_NUM       float64
Current Rating_NUM                     float64
Joule-integral-Nom (J)_NUM             float64
Maximum AC Voltage Rating_NUM          float64
Maximum DC Voltage Rating_NUM          float64
Maximum Power Dissipation_NUM          float64
Operating Temperature-Max (Cel)_NUM    float64
Operating Temperature-Min (Cel)_NUM    float64
Pre-arcing time-Min (ms)_NUM           float64
Product Diameter_NUM                   float64
Product Length_NUM                     float64
Rated Breaking Capacity (A)_NUM        float64
Rated Current (A)_NUM                  float64
Rated Voltage (V)_NUM                  float64
Rated Voltage(AC) (V)_NUM              float64
Rated Voltage(DC) (V)_NUM              float64
dtype: object

### Filling Missing Values

In [13]:
# Finding 2: High Proportion of Missing Values (Sparsity)
categorical_features = [
    'Acting', 'Additional Feature', 'Application', 'Blow Characteristic',
    'Fuse Material', 'Fuse Size', 'JESD-609 Code', 'LC Risk',
    'Mounting', 'Mounting Feature', 'Number of Terminals'
]

# Imputation:
# 1. Numerical features (newly created) are imputed with the median
for col in numeric_features:
    median_val = df_cleaned[col].median()
    df_cleaned[col] = df_cleaned[col].fillna(median_val)

# 2. Categorical/Text features are imputed with a placeholder
for col in categorical_features:
    df_cleaned[col] = df_cleaned[col].fillna('Unknown')
df_cleaned['PART_DESCRIPTION'] = df_cleaned['PART_DESCRIPTION'].fillna('no description provided')

In [15]:
df_cleaned[numeric_features+categorical_features].info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 998 entries, 0 to 997
Data columns (total 29 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Body Breadth (mm)_NUM                998 non-null    float64
 1   Body Height (mm)_NUM                 998 non-null    float64
 2   Body Length or Diameter (mm)_NUM     998 non-null    float64
 3   Current Rating_NUM                   998 non-null    float64
 4   Joule-integral-Nom (J)_NUM           998 non-null    float64
 5   Maximum AC Voltage Rating_NUM        998 non-null    float64
 6   Maximum DC Voltage Rating_NUM        998 non-null    float64
 7   Maximum Power Dissipation_NUM        998 non-null    float64
 8   Operating Temperature-Max (Cel)_NUM  998 non-null    float64
 9   Operating Temperature-Min (Cel)_NUM  998 non-null    float64
 10  Pre-arcing time-Min (ms)_NUM         998 non-null    float64
 11  Product Diameter_NUM            

## Model to identify similar materials based on "Part Description" 

In [16]:
df_model_text = df_cleaned[df_cleaned['PART_DESCRIPTION'] != 'no description provided'].copy().reset_index(drop=True)

In [18]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_model_text['PART_DESCRIPTION'])

print("--- Text Similarity Model Setup ---")
print(f"Dataset Size for Text Model: {len(df_model_text)} entries")
print(f"TF-IDF Matrix Shape (Entries x Unique Words): {tfidf_matrix.shape}")

--- Text Similarity Model Setup ---
Dataset Size for Text Model: 663 entries
TF-IDF Matrix Shape (Entries x Unique Words): (663, 207)


In [20]:
# Cosine Similarity Calculation
cosine_sim_text = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a mapping series to quickly look up the internal index from the PART_ID
part_to_index_text = pd.Series(df_model_text.index, index=df_model_text['PART_ID']).drop_duplicates()


In [21]:
def get_similar_parts_text(part_id, n=5, df_data=df_model_text, part_to_index=part_to_index_text, cosine_sim=cosine_sim_text):
    """
    Finds the top N most similar parts based only on PART_DESCRIPTION.
    """
    if part_id not in part_to_index:
        return f"Part ID '{part_id}' not found in the descriptive subset."

    # Get the index of the part that matches the ID
    idx = part_to_index[part_id]

    # Get the similarity scores for that part with all other parts
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the scores and exclude the first one (the part itself, which has score 1.0)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]

    # Get the indices and scores of the top alternatives
    part_indices = [i[0] for i in sim_scores]
    scores = [f"{i[1]:.4f}" for i in sim_scores]

    # Compile and return the results as a DataFrame
    return pd.DataFrame({
        'Alternative PART_ID': df_data['PART_ID'].iloc[part_indices],
        'Alternative PART_DESCRIPTION': df_data['PART_DESCRIPTION'].iloc[part_indices],
        'Similarity Score': scores
    }).reset_index(drop=True)

In [22]:
sample_part_id = 'A1' # First part in the descriptive subset
similar_parts_df = get_similar_parts_text(sample_part_id, n=5)

print("\n--- Sample Execution: Top 5 Alternatives for PART_ID: A1 (Text-Only Model) ---")
print(f"Target Description: {df_model_text[df_model_text['PART_ID'] == sample_part_id]['PART_DESCRIPTION'].iloc[0]}")
print(similar_parts_df.to_markdown(index=False, numalign="left", stralign="left"))


--- Sample Execution: Top 5 Alternatives for PART_ID: A1 (Text-Only Model) ---
Target Description: Fuse Miniature Fast Acting 1.6A 250V Holder Cartridge 5 X 20mm Ceramic Box CCC/PSE/VDE/cULus Electric Fuse, Very Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm
| Alternative PART_ID   | Alternative PART_DESCRIPTION                                                                                                                                                                            | Similarity Score   |
|:----------------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------------------|
| A253                  | Fuse Miniature Fast Acting 1.6A 250V Holder Cartridge 5 X 20mm Ceramic Bulk CCC/CE/CSA/KC/PSE/SEMKO/UL/VDE Electric Fuse, Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm                    | 0.

## Steps to expand the model to account for all the other attributes within the dataset

- Data Preparation for Non-Text Attributes
  - Before combining the data, all non-text attributes (the numerical and categorical columns) must be cleaned, imputed, and prepared as vectors.
    - Imputation check
    - Numerical scaling
- Categorical features vectorization
  - One-hot encoding
- Feature stacking and Unified matrix creation
  - Combine scaled numerical features, one-hot encoded categorical features and TF-IDF matrix into a single unified feature. This final matrix will represent every fuse as a single, multi-dimensional vector encompassing all its characteristics.    